In [ ]:
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler



X_train = pd.read_csv("../data/processed/train/X_train.csv")
X_test = pd.read_csv("../data/processed/test/X_test.csv")

y_train = pd.read_csv("../data/processed/train/y_train.csv").squeeze("columns")
y_test = pd.read_csv("../data/processed/test/y_test.csv").squeeze("columns")

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
ordinal_cols = ["Physical_Activity", "Diet_Quality"]
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns.drop(ordinal_cols)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[
            ["Low", "Moderate", "High"],
            ["Poor", "Average", "Good"]
        ],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("ordinal", ordinal_pipeline, ordinal_cols),
    ("cat", categorical_pipeline, cat_cols)
])